In [ ]:
# Cell 0: Kaggle Auth — set your Kaggle API token here
import os, json

KAGGLE_TOKEN = "KGAT_0a9a064c90b6d6ac2dc57999527ed379"
KAGGLE_USER  = "peroplayer"

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USER, "key": KAGGLE_TOKEN}, f)
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
print(f"Authenticated as {KAGGLE_USER}")

In [ ]:
# Cell 1: Setup — install deps, clone repo, clear old checkpoints
import os, subprocess

print("Cloning Talos V1 repo...")

REPO = "https://github.com/vfxjamer/talonbot.git"
DATASET = "peroplayer/talon-v1-checkpoints"

!rm -rf /kaggle/working/talonbot
!git clone {REPO} /kaggle/working/talonbot
print("Project cloned.")

# Restore checkpoints from Kaggle dataset
ckpt_dir = "/kaggle/working/talonbot/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
ret = subprocess.run(
    ["kaggle", "datasets", "download", DATASET, "--path", ckpt_dir, "--unzip", "--force"],
    capture_output=True, text=True
)
if ret.returncode == 0:
    print(f"Checkpoints restored from Kaggle dataset {DATASET}")
else:
    print(f"No prior checkpoints on Kaggle ({ret.stderr.strip()})")

# Clear checkpoints if architecture changed (new 1024x4 network is incompatible with old 256x2)
latest = 0
if os.path.exists(ckpt_dir):
    for d in os.listdir(ckpt_dir):
        p = os.path.join(ckpt_dir, d)
        if os.path.isdir(p) and d.isdigit():
            latest = max(latest, int(d))
    # If checkpoints are from old arch (< 1M steps), clear them
    if latest > 0 and latest < 1_000_000:
        print(f"Old-architecture checkpoints detected ({latest:,} steps). Clearing for new 1024x4 network...")
        import shutil
        for d in os.listdir(ckpt_dir):
            p = os.path.join(ckpt_dir, d)
            if os.path.isdir(p) and d.isdigit():
                shutil.rmtree(p)

# Run setup with full output
ret = subprocess.run(["bash", "/kaggle/working/talonbot/setup.sh"], capture_output=True, text=True)
if ret.returncode != 0:
    print("=== FULL STDOUT ===")
    print(ret.stdout[-8000:] if len(ret.stdout) > 8000 else ret.stdout)
    print("=== STDERR ===")
    print(ret.stderr[-2000:] if len(ret.stderr) > 2000 else ret.stderr)
    raise RuntimeError(f"Setup failed (exit {ret.returncode})")
else:
    print(ret.stdout[-2000:] if len(ret.stdout) > 2000 else ret.stdout)
print("Setup done.")

In [ ]:
# Cell 2: Build TalonBot (1024x4 V4 network, uses system torch after cu124 reinstall)
import os, torch

os.environ["GIGALEARN_ROOT"] = "/workspace/libs/GigaLearnCPP"
torch_cmake = torch.utils.cmake_prefix_path + "/Torch"
print(f"Torch cmake: {torch_cmake}")

# Ensure latest code (in case Cell 2 was run without re-running Cell 1)
!git -C /kaggle/working/talonbot pull 2>&1

cmake_cmd = [
    "cmake", "..",
    "-DCMAKE_BUILD_TYPE=RelWithDebInfo",
    "-DCMAKE_POSITION_INDEPENDENT_CODE=ON",
    f"-DGIGALEARN_ROOT=/workspace/libs/GigaLearnCPP",
    f"-DTorch_DIR={torch_cmake}"
]

!cd /kaggle/working/talonbot && rm -rf build && mkdir build && cd build && \
  {' '.join(cmake_cmd)} 2>&1 && \
  cmake --build . --config Release --target TalonBot -j$(nproc) 2>&1
print("Build done.")

In [ ]:
# Cell 2.5: Incremental rebuild (skips cmake, just recompiles changed files)
!cd /kaggle/working/talonbot && git pull 2>&1
!ln -sfn ObsBuilders /workspace/libs/GigaLearnCPP/RLGymCPP/src/RLGymCPP/OBSBuilders 2>/dev/null; sync
!cd /kaggle/working/talonbot/build && cmake --build . --config Release --target TalonBot -j$(nproc) 2>&1
print("Rebuild done.")

In [ ]:
# Cell 3: Run training + backup daemon (checkpoints pushed to Kaggle dataset every 60min)
import subprocess, sys, os, json, time, threading

CWD = "/kaggle/working/talonbot"
DATASET = "peroplayer/talon-v1-checkpoints"
CKPT_DIR = os.path.join(CWD, "checkpoints")

print(f"{'='*50}")
print(f"  Starting Talos V1 — V4 Rewards, 1024x4 Network")
print(f"  Phase will be auto-detected from checkpoints")
print(f"  Backup to Kaggle dataset: {DATASET}")
print(f"{'='*50}")

# Kaggle dataset metadata
os.makedirs(CKPT_DIR, exist_ok=True)
meta_path = os.path.join(CKPT_DIR, "dataset-metadata.json")
if not os.path.exists(meta_path):
    with open(meta_path, "w") as f:
        json.dump({"title": "Talos V1 Checkpoints", "id": DATASET, "licenses": [{"name": "CC0-1.0"}]}, f)

def backup_loop():
    while True:
        time.sleep(3600)
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", CKPT_DIR, "-m", "checkpoint", "--dir-mode", "zip"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"[BACKUP] Pushed to {DATASET} at {time.strftime('%H:%M:%S UTC', time.gmtime())}")
        else:
            err = result.stderr.strip()
            if "404" in err or "Not Found" in err:
                subprocess.run(["kaggle", "datasets", "create", "-p", CKPT_DIR, "--public"], capture_output=True)
                print(f"[BACKUP] Created dataset {DATASET}")
            else:
                print(f"[BACKUP] {err}")

t = threading.Thread(target=backup_loop, daemon=True)
t.start()
print(f"[BACKUP] Daemon started — pushes {DATASET} every 60min")

p = subprocess.Popen(
    [os.path.join(CWD, "build", "TalonBot"), "/workspace/libs/collision_meshes"],
    cwd=CWD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True
)
for line in p.stdout:
    print(line, end=""); sys.stdout.flush()
p.wait()
print(f"\nExit code: {p.returncode}")